In [0]:
# Notebook responsável pela transformação da camada Silver para Gold.

from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

print("Ambiente Gold preparado com sucesso.")

In [0]:
# Carrega as tabelas tratadas da camada Silver.

df_info = spark.table("workspace.silver.tb_info_filmes")
df_financeiro = spark.table("workspace.silver.tb_financeiro_filmes")
df_metricas = spark.table("workspace.silver.tb_metricas_engajamento")
df_avaliacoes = spark.table("workspace.silver.tb_avaliacoes_usuarios")
df_generos = spark.table("workspace.silver.tb_generos")
df_pessoas_empresas = spark.table("workspace.silver.tb_pessoas_empresas")
df_cotacao = spark.table("workspace.silver.tb_cotacao_dolar")

print("Tabelas Silver carregadas com sucesso.")


In [0]:
# Cria e grava a dimensão de filmes na Gold

from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window

janela_filmes = Window.orderBy("id_filme")

dim_movies = (
    df_info
    .select(
        "id_filme",
        "titulo",
        "data_lancamento",
        "ano_lancamento",
        "duracao_minutos",
        "idioma_original",
        "status_filme",
        "sinopse"
    )
    .dropDuplicates(["id_filme"])
    .withColumn(
        "sk_movie_id",
        row_number().over(janela_filmes).cast("bigint")
    )
    .select(
        "sk_movie_id",
        col("id_filme").cast("string").alias("id_filme"),
        col("titulo").cast("string").alias("titulo"),
        col("data_lancamento").cast("date").alias("data_lancamento"),
        col("ano_lancamento").cast("int").alias("ano_lancamento"),
        col("duracao_minutos").cast("int").alias("duracao_minutos"),
        col("idioma_original").cast("string").alias("idioma_original"),
        col("status_filme").cast("string").alias("status_filme"),
        col("sinopse").cast("string").alias("sinopse")
    )
)

(
    dim_movies.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.dim_movies")
)

print(
    "Dimensão de filmes gravada com sucesso:",
    spark.table("workspace.gold.dim_movies").count(),
    "registros"
)

display(spark.table("workspace.gold.dim_movies").limit(20))



In [0]:
# Cria e grava a dimensão de avaliações dos usuários

from pyspark.sql.functions import (
    col, count, avg, round as spark_round, row_number
)
from pyspark.sql.window import Window

avaliacoes_por_filme = (
    df_avaliacoes.alias("a")
    .join(
        dim_movies.alias("m"),
        col("a.id_filme") == col("m.id_filme"),
        "inner"
    )
    .groupBy(col("m.sk_movie_id"))
    .agg(
        count(col("a.id_filme")).cast("int").alias("qtd_avaliacoes_usuarios"),
        spark_round(
            avg(col("a.nota_usuario")), 2
        ).cast("double").alias("nota_media_usuarios")
    )
)

dim_reviews = (
    avaliacoes_por_filme
    .withColumn(
        "sk_review_id",
        row_number().over(
            Window.orderBy("sk_movie_id")
        ).cast("bigint")
    )
    .select(
        "sk_review_id",
        "sk_movie_id",
        "qtd_avaliacoes_usuarios",
        "nota_media_usuarios"
    )
)

(
    dim_reviews.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.dim_reviews")
)

print(
    "Dimensão de avaliações gravada:",
    spark.table("workspace.gold.dim_reviews").count(),
    "registros"
)

display(dim_reviews.limit(10))



In [0]:
# Cria a dimensão de gêneros.

janela_generos = Window.orderBy("genero")

dim_genres = (
    df_generos
    .select("genero")
    .filter(
        col("genero").isNotNull() &
        (trim(col("genero")) != "")
    )
    .dropDuplicates(["genero"])
    .withColumn(
        "sk_genre_id",
        row_number().over(janela_generos).cast("bigint")
    )
    .select(
        "sk_genre_id",
        col("genero").cast("string").alias("nome_genero")
    )
)

display(dim_genres.orderBy("sk_genre_id"))


In [0]:
# Cria e grava a dimensão de pessoas

from pyspark.sql.functions import col, trim, length, row_number
from pyspark.sql.window import Window

df_people_base = (
    df_pessoas_empresas
    .filter(col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .select(
        trim(col("nome_entidade")).alias("nome_pessoa"),
        col("tipo_entidade").alias("tipo_pessoa")
    )
    .filter(
        col("nome_pessoa").isNotNull() &
        (col("nome_pessoa") != "") &
        (length(col("nome_pessoa")) <= 100)
    )
    .dropDuplicates(["nome_pessoa", "tipo_pessoa"])
)

dim_people = (
    df_people_base
    .withColumn(
        "sk_person_id",
        row_number().over(
            Window.orderBy("nome_pessoa", "tipo_pessoa")
        ).cast("bigint")
    )
    .select("sk_person_id", "nome_pessoa", "tipo_pessoa")
)

(
    dim_people.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.dim_people")
)

print(
    "Dimensão de pessoas gravada:",
    spark.table("workspace.gold.dim_people").count(),
    "registros"
)

display(dim_people.limit(20))



In [0]:
# Cria e grava a dimensão de empresas

from pyspark.sql.functions import col, trim, row_number
from pyspark.sql.window import Window

df_empresas_base = (
    df_pessoas_empresas
    .filter(col("tipo_entidade") == "Produtora")
    .select(trim(col("nome_entidade")).alias("nome_produtora"))
    .filter(
        col("nome_produtora").isNotNull() &
        (col("nome_produtora") != "")
    )
    .dropDuplicates(["nome_produtora"])
)

janela_empresas = Window.orderBy("nome_produtora")

dim_companies = (
    df_empresas_base
    .withColumn(
        "sk_company_id",
        row_number().over(janela_empresas).cast("bigint")
    )
    .select("sk_company_id", "nome_produtora")
)

(
    dim_companies.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.dim_companies")
)

print(
    "Dimensão de empresas gravada com sucesso:",
    spark.table("workspace.gold.dim_companies").count(),
    "registros"
)

display(spark.table("workspace.gold.dim_companies").limit(10))



In [0]:
# Grava a dimensão de gêneros e cria a ligação com filmes

from pyspark.sql.functions import col, trim

(
    dim_genres.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.dim_genres")
)

df_filmes_generos = spark.table("workspace.silver.tb_generos")

bridge_movie_genre = (
    df_filmes_generos.alias("fg")
    .join(
        dim_movies.alias("f"),
        trim(col("fg.id_filme")) == trim(col("f.id_filme")),
        "inner"
    )
    .join(
        dim_genres.alias("g"),
        trim(col("fg.genero")) == trim(col("g.nome_genero")),
        "inner"
    )
    .select(
        col("f.sk_movie_id"),
        col("g.sk_genre_id")
    )
    .dropDuplicates()
)

(
    bridge_movie_genre.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.bridge_movie_genre")
)

print(
    "Dimensão de gêneros:",
    spark.table("workspace.gold.dim_genres").count(),
    "registros"
)

print(
    "Ligação entre filmes e gêneros:",
    spark.table("workspace.gold.bridge_movie_genre").count(),
    "registros"
)

display(spark.table("workspace.gold.bridge_movie_genre").limit(10))



In [0]:
# Cria a ligação entre filmes e pessoas

from pyspark.sql.functions import col, trim

bridge_movie_person = (
    df_pessoas_empresas.alias("c")
    .filter(col("c.tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .join(
        dim_movies.alias("f"),
        trim(col("c.id_filme")) == trim(col("f.id_filme")),
        "inner"
    )
    .join(
        dim_people.alias("p"),
        (trim(col("c.nome_entidade")) == trim(col("p.nome_pessoa"))) &
        (col("c.tipo_entidade") == col("p.tipo_pessoa")),
        "inner"
    )
    .select(
        col("f.sk_movie_id"),
        col("p.sk_person_id")
    )
    .dropDuplicates()
)

(
    bridge_movie_person.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.bridge_movie_person")
)

print(
    "Ligação entre filmes e pessoas:",
    spark.table("workspace.gold.bridge_movie_person").count(),
    "registros"
)

display(spark.table("workspace.gold.bridge_movie_person").limit(10))



In [0]:
# Cria a ligação entre filmes e empresas

from pyspark.sql.functions import col, trim

bridge_movie_company = (
    df_pessoas_empresas.alias("c")
    .filter(col("c.tipo_entidade") == "Produtora")
    .join(
        dim_movies.alias("f"),
        trim(col("c.id_filme")) == trim(col("f.id_filme")),
        "inner"
    )
    .join(
        dim_companies.alias("e"),
        trim(col("c.nome_entidade")) == trim(col("e.nome_produtora")),
        "inner"
    )
    .select(
        col("f.sk_movie_id"),
        col("e.sk_company_id")
    )
    .dropDuplicates()
)

(
    bridge_movie_company.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.bridge_movie_company")
)

print(
    "Ligação entre filmes e empresas:",
    spark.table("workspace.gold.bridge_movie_company").count(),
    "registros"
)

display(spark.table("workspace.gold.bridge_movie_company").limit(10))



In [0]:
# Cria a tabela fato de desempenho dos filmes

from pyspark.sql.functions import col, row_number, desc_nulls_last, current_date
from pyspark.sql.window import Window

# Mantém um registro financeiro por filme para evitar duplicações nos joins.
janela_financeiro = Window.partitionBy("id_filme").orderBy(
    desc_nulls_last("ingestion_datetime")
)

df_financeiro_unico = (
    df_financeiro
    .withColumn("rn", row_number().over(janela_financeiro))
    .filter(col("rn") == 1)
    .drop("rn", "ingestion_datetime")
)

# Mantém um registro de métricas por filme.
janela_metricas = Window.partitionBy("id_filme").orderBy(
    desc_nulls_last("ingestion_datetime")
)

df_metricas_unicas = (
    df_metricas
    .withColumn("rn", row_number().over(janela_metricas))
    .filter(col("rn") == 1)
    .drop("rn", "ingestion_datetime")
)

# A tabela fato considera apenas filmes lançados até a data atual.
fact_movies_performance = (
    dim_movies.alias("f")
    .filter(
        (col("f.status_filme") == "Lançado") &
        col("f.data_lancamento").isNotNull() &
        (col("f.data_lancamento") <= current_date())
    )
    .join(
        df_financeiro_unico.alias("fin"),
        col("f.id_filme") == col("fin.id_filme"),
        "left"
    )
    .join(
        df_metricas_unicas.alias("m"),
        col("f.id_filme") == col("m.id_filme"),
        "left"
    )
    .select(
        col("f.sk_movie_id").cast("bigint").alias("sk_movie_id"),
        col("fin.orcamento_usd").cast("decimal(18,2)").alias("orcamento_usd"),
        col("fin.receita_usd").cast("decimal(18,2)").alias("receita_usd"),
        col("fin.lucro_usd").cast("decimal(18,2)").alias("lucro_usd"),
        col("fin.orcamento_brl").cast("decimal(18,2)").alias("orcamento_brl"),
        col("fin.receita_brl").cast("decimal(18,2)").alias("receita_brl"),
        col("fin.lucro_brl").cast("decimal(18,2)").alias("lucro_brl"),
        col("m.popularidade").cast("double").alias("popularidade"),
        col("m.nota_media_tmdb").cast("double").alias("nota_media_tmdb"),
        col("m.qtd_votos_tmdb").cast("int").alias("qtd_votos_tmdb"),
        col("m.nota_media_imdb").cast("double").alias("nota_media_imdb"),
        col("m.qtd_votos_imdb").cast("int").alias("qtd_votos_imdb")
    )
)

(
    fact_movies_performance.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.fact_movies_performance")
)

print(
    "Tabela fato gravada com sucesso:",
    spark.table("workspace.gold.fact_movies_performance").count(),
    "registros"
)

display(spark.table("workspace.gold.fact_movies_performance").limit(10))



In [0]:
# Calcula a receita total dos filmes em reais

from pyspark.sql.functions import col, sum as spark_sum, round as spark_round

resultado_faturamento = (
    spark.table("workspace.gold.fact_movies_performance")
    .agg(
        spark_round(
            spark_sum(col("receita_brl")), 2
        ).alias("receita_total_brl")
    )
)

display(resultado_faturamento)



In [0]:
# Exibe os 5 filmes com maior popularidade

from pyspark.sql.functions import col

top_5_popularidade = (
    spark.table("workspace.gold.fact_movies_performance").alias("f")
    .join(
        spark.table("workspace.gold.dim_movies").alias("m"),
        col("f.sk_movie_id") == col("m.sk_movie_id"),
        "inner"
    )
    .filter(col("f.popularidade").isNotNull())
    .select(
        col("m.titulo"),
        col("f.popularidade")
    )
    .orderBy(col("popularidade").desc(), col("titulo").asc())
    .limit(5)
)

display(top_5_popularidade)



In [0]:
# Conta a quantidade de filmes por gênero

from pyspark.sql.functions import col, countDistinct

filmes_por_genero = (
    spark.table("workspace.gold.bridge_movie_genre").alias("b")
    .join(
        spark.table("workspace.gold.dim_genres").alias("g"),
        col("b.sk_genre_id") == col("g.sk_genre_id"),
        "inner"
    )
    .groupBy(col("g.nome_genero"))
    .agg(
        countDistinct("b.sk_movie_id").alias("quantidade_filmes")
    )
    .orderBy(
        col("quantidade_filmes").desc(),
        col("nome_genero").asc()
    )
)

display(filmes_por_genero)



In [0]:
# Exibe os 10 filmes com maior faturamento

from pyspark.sql.functions import col, rank
from pyspark.sql.window import Window

janela_ranking = Window.orderBy(col("receita_usd").desc())

top_10_faturamento = (
    spark.table("workspace.gold.fact_movies_performance").alias("f")
    .join(
        spark.table("workspace.gold.dim_movies").alias("m"),
        col("f.sk_movie_id") == col("m.sk_movie_id"),
        "inner"
    )
    .filter(col("f.receita_usd").isNotNull())
    .select(
        col("m.titulo"),
        col("f.receita_usd"),
        col("f.receita_brl")
    )
    .withColumn(
        "posicao_ranking",
        rank().over(janela_ranking)
    )
    .orderBy(col("posicao_ranking").asc(), col("titulo").asc())
    .limit(10)
    .select(
        "posicao_ranking",
        "titulo",
        "receita_usd",
        "receita_brl"
    )
)

display(top_10_faturamento)



In [0]:
# Identifica o ator com mais filmes nos últimos dois anos

from pyspark.sql.functions import (
    col, countDistinct, current_date, add_months,
    max as spark_max, lit
)

df_filmes = spark.table("workspace.gold.dim_movies")

# Usa a data de lançamento mais recente da base, sem filmes futuros ou não lançados.
data_referencia = (
    df_filmes
    .filter(
        col("data_lancamento").isNotNull() &
        (col("data_lancamento") <= current_date()) &
        (col("status_filme") == "Lançado")
    )
    .agg(spark_max("data_lancamento").alias("data_maxima"))
    .first()["data_maxima"]
)

if data_referencia is None:
    print("Não há filmes lançados com data válida para esta análise.")
else:
    participacoes_atores = (
        spark.table("workspace.gold.bridge_movie_person").alias("b")
        .join(
            df_filmes.alias("m"),
            col("b.sk_movie_id") == col("m.sk_movie_id"),
            "inner"
        )
        .join(
            spark.table("workspace.gold.dim_people").alias("p"),
            col("b.sk_person_id") == col("p.sk_person_id"),
            "inner"
        )
        .filter(
            (col("p.tipo_pessoa") == "Ator") &
            (col("m.data_lancamento") >= add_months(lit(data_referencia), -24)) &
            (col("m.data_lancamento") <= lit(data_referencia)) &
            (col("m.status_filme") == "Lançado")
        )
        .groupBy(
            col("p.sk_person_id"),
            col("p.nome_pessoa")
        )
        .agg(
            countDistinct("b.sk_movie_id").alias("quantidade_filmes")
        )
        .orderBy(
            col("quantidade_filmes").desc(),
            col("nome_pessoa").asc()
        )
        .limit(1)
        .select(
            col("nome_pessoa"),
            col("quantidade_filmes")
        )
    )

    print("Data de referência:", data_referencia)
    display(participacoes_atores)

    

In [0]:
# Identifica a produtora com maior lucro nos últimos cinco anos

from pyspark.sql.functions import (
    col, countDistinct, sum as spark_sum,
    current_date, add_months, max as spark_max, lit
)

df_filmes = spark.table("workspace.gold.dim_movies")

# Usa a data de lançamento mais recente, desconsiderando filmes futuros ou não lançados.

data_referencia = (
    df_filmes
    .filter(
        col("data_lancamento").isNotNull() &
        (col("data_lancamento") <= current_date()) &
        (col("status_filme") == "Lançado")
    )
    .agg(spark_max("data_lancamento").alias("data_maxima"))
    .first()["data_maxima"]
)

if data_referencia is None:
    print("Não há filmes lançados com data válida para esta análise.")
else:
    lucro_empresas = (
        spark.table("workspace.gold.bridge_movie_company").alias("b")
        .join(
            spark.table("workspace.gold.dim_companies").alias("c"),
            col("b.sk_company_id") == col("c.sk_company_id"),
            "inner"
        )
        .join(
            df_filmes.alias("m"),
            col("b.sk_movie_id") == col("m.sk_movie_id"),
            "inner"
        )
        .join(
            spark.table("workspace.gold.fact_movies_performance").alias("f"),
            col("b.sk_movie_id") == col("f.sk_movie_id"),
            "inner"
        )
        .filter(
            (col("m.data_lancamento") >= add_months(lit(data_referencia), -60)) &
            (col("m.data_lancamento") <= lit(data_referencia)) &
            (col("m.status_filme") == "Lançado") &
            col("f.lucro_usd").isNotNull()
        )
        .groupBy(
            col("c.sk_company_id"),
            col("c.nome_produtora")
        )
        .agg(
            countDistinct("b.sk_movie_id").alias("quantidade_filmes"),
            spark_sum(col("f.lucro_usd"))
                .cast("decimal(18,2)")
                .alias("lucro_total_usd")
        )
        .orderBy(
            col("lucro_total_usd").desc(),
            col("nome_produtora").asc()
        )
        .limit(1)
        .select(
            "nome_produtora",
            "quantidade_filmes",
            "lucro_total_usd"
        )
    )

    print("Data de referência:", data_referencia)
    display(lucro_empresas)



In [0]:
# Cria a tabela de contexto dos filmes para GenAI

from pyspark.sql.functions import (
    col, collect_set, concat_ws, concat, lit, coalesce,
    when, trim, format_number, sort_array
)

df_filmes = spark.table("workspace.gold.dim_movies")
df_fato = spark.table("workspace.gold.fact_movies_performance")
df_ponte_pessoas = spark.table("workspace.gold.bridge_movie_person")
df_pessoas = spark.table("workspace.gold.dim_people")

# Reúne os atores e diretores de cada filme.
pessoas_por_filme = (
    df_ponte_pessoas.alias("b")
    .join(
        df_pessoas.alias("p"),
        col("b.sk_person_id") == col("p.sk_person_id"),
        "inner"
    )
    .groupBy(col("b.sk_movie_id"))
    .agg(
        concat_ws(
            ", ",
            sort_array(
                collect_set(
                    when(
                        col("p.tipo_pessoa") == "Ator",
                        col("p.nome_pessoa")
                    )
                )
            )
        ).alias("atores"),
        concat_ws(
            ", ",
            sort_array(
                collect_set(
                    when(
                        col("p.tipo_pessoa") == "Diretor",
                        col("p.nome_pessoa")
                    )
                )
            )
        ).alias("diretores")
    )
)

# Usa valores substitutos para evitar que campos nulos anulem o texto.
gold_genai_movies_context = (
    df_filmes.alias("m")
    .join(
        df_fato.alias("f"),
        col("m.sk_movie_id") == col("f.sk_movie_id"),
        "left"
    )
    .join(
        pessoas_por_filme.alias("p"),
        col("m.sk_movie_id") == col("p.sk_movie_id"),
        "left"
    )
    .select(
        col("m.id_filme").alias("movie_id"),
        col("m.titulo").alias("title"),
        concat(
            lit("O filme "),
            coalesce(
                col("m.titulo"),
                lit("de título não informado")
            ),
            lit(", lançado no ano de "),
            coalesce(
                col("m.ano_lancamento").cast("string"),
                lit("ano não informado")
            ),
            lit(", faturou "),
            coalesce(
                concat(
                    lit("US$ "),
                    format_number(col("f.receita_usd"), 2)
                ),
                lit("um valor não informado")
            ),
            lit(" e teve um custo de "),
            coalesce(
                concat(
                    lit("US$ "),
                    format_number(col("f.orcamento_usd"), 2)
                ),
                lit("valor não informado")
            ),
            lit(". Estrelado por "),
            when(
                col("p.atores").isNotNull() &
                (trim(col("p.atores")) != ""),
                col("p.atores")
            ).otherwise(lit("atores não informados")),
            lit(" e dirigido por "),
            when(
                col("p.diretores").isNotNull() &
                (trim(col("p.diretores")) != ""),
                col("p.diretores")
            ).otherwise(lit("diretor não informado")),
            lit(", o filme possui a seguinte sinopse: "),
            when(
                col("m.sinopse").isNotNull() &
                (trim(col("m.sinopse")) != ""),
                col("m.sinopse")
            ).otherwise(lit("Sinopse não informada")),
            lit(".")
        ).alias("llm_context_document")
    )
)

(
    gold_genai_movies_context.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.gold_genai_movies_context")
)

print(
    "Tabela de contexto GenAI gravada:",
    spark.table("workspace.gold.gold_genai_movies_context").count(),
    "registros"
)

display(
    spark.table("workspace.gold.gold_genai_movies_context").limit(10)
)

